# 📝 OCR MANUSCRITO - Transfer Learning

**Dataset:** IAM Handwriting Database  
**Estrategia:** Transfer learning desde modelo impreso  
**Tiempo:** 30-50 horas (Colab Free)  

---

## ⚠️ REQUISITOS:

1. Modelo impreso entrenado (`ocr_model_printed_final.pth`) en Drive
2. Dataset IAM descargado manualmente y subido a Drive
3. Registro en: https://fki.tic.heia-fr.ch/databases/iam-handwriting-database

---

In [ ]:
# ============================================================
# CELDA 1: Setup GPU
# ============================================================
import torch
import sys

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# CELDA 2: Montar Google Drive
# ============================================================
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/OCR_Project')
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints_handwriting'
DRIVE_MODELS = DRIVE_ROOT / 'models'
DRIVE_DATASETS = DRIVE_ROOT / 'datasets'
DRIVE_RESULTS = DRIVE_ROOT / 'results'

for folder in [DRIVE_CHECKPOINTS, DRIVE_MODELS, DRIVE_DATASETS, DRIVE_RESULTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Drive montado")

In [ ]:
# ============================================================
# CELDA 3: Instalación
# ============================================================
!pip install -q opencv-python-headless pillow matplotlib albumentations editdistance tqdm python-Levenshtein

print("✅ Instalado")

In [ ]:
# ============================================================
# CELDA 4: Imports
# ============================================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import editdistance
import albumentations as A
from albumentations.pytorch import ToTensorV2
import string
import random
import json
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("✅ Imports")

In [ ]:
# ============================================================
# CELDA 5: Configuración Manuscrito
# ============================================================

class Config:
    # Vocabulario (mismo que impreso)
    CHARS = (
        string.ascii_letters +
        string.digits +
        string.punctuation +
        ' ' +
        'áéíóúÁÉÍÓÚñÑüÜ¿¡'
    )

    CHAR_TO_IDX = {char: idx + 1 for idx, char in enumerate(CHARS)}
    IDX_TO_CHAR = {idx: char for char, idx in CHAR_TO_IDX.items()}
    IDX_TO_CHAR[0] = '<blank>'
    NUM_CLASSES = len(CHAR_TO_IDX) + 1

    # Arquitectura
    IMG_HEIGHT = 64
    IMG_WIDTH = 256
    HIDDEN_SIZE = 512
    NUM_LSTM_LAYERS = 3

    # Training manuscrito
    BATCH_SIZE = 32  # Reducido
    NUM_WORKERS = 2
    LEARNING_RATE = 0.00005  # MUY bajo
    WEIGHT_DECAY = 1e-5

    MAX_EPOCHS = 100
    EARLY_STOP_PATIENCE = 25
    SAVE_EVERY_N_EPOCHS = 5
    KEEP_LAST_N_CHECKPOINTS = 3

    # Paths
    CHECKPOINT_DIR = DRIVE_CHECKPOINTS
    MODEL_DIR = DRIVE_MODELS
    RESULTS_DIR = DRIVE_RESULTS
    IAM_DIR = DRIVE_DATASETS / 'IAM'

    # Transfer learning
    PRINTED_MODEL = DRIVE_MODELS / 'ocr_model_printed_final.pth'
    FREEZE_CNN = True
    UNFREEZE_EPOCH = 15

config = Config()

print(f"✅ Config manuscrito:")
print(f"   Batch: {config.BATCH_SIZE}")
print(f"   LR: {config.LEARNING_RATE}")
print(f"   Transfer desde: {config.PRINTED_MODEL.name}")

In [ ]:
# ============================================================
# CELDA 6: Utilidades (iguales que impreso)
# ============================================================

def encode_text(text):
    encoded = []
    for char in text:
        if char in config.CHAR_TO_IDX:
            encoded.append(config.CHAR_TO_IDX[char])
    return encoded if encoded else [0]

def decode_prediction(indices):
    chars = []
    prev_idx = -1
    for idx in indices:
        if idx != 0 and idx != prev_idx:
            char = config.IDX_TO_CHAR.get(idx, '')
            if char and char != '<blank>':
                chars.append(char)
        prev_idx = idx
    return ''.join(chars)

def calculate_cer(pred, truth):
    if len(truth) == 0:
        return 0.0 if len(pred) == 0 else 1.0
    return editdistance.eval(pred, truth) / len(truth)

def calculate_wer(pred, truth):
    pred_words = pred.split()
    truth_words = truth.split()
    if len(truth_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0
    return editdistance.eval(pred_words, truth_words) / len(truth_words)

def calculate_accuracy(pred, truth):
    return 1.0 if pred.strip() == truth.strip() else 0.0

print("✅ Utilidades")

---

## 📥 DATASET IAM

**INSTRUCCIONES:**

1. Regístrate: https://fki.tic.heia-fr.ch/databases/iam-handwriting-database
2. Descarga:
   - `lines.tgz` (~1.2GB)
   - `ascii/lines.txt`
3. Sube a Drive: `/MyDrive/OCR_Project/datasets/IAM/`
4. Ejecuta CELDA 7 para extraer

---

In [ ]:
# ============================================================
# CELDA 7: Extraer IAM (Soporte para múltiples archivos .tgz)
# ============================================================
import tarfile

print(f"Buscando archivos .tgz en: {config.IAM_DIR}")

# Buscar todos los archivos .tgz
tgz_files = list(config.IAM_DIR.glob('*.tgz'))

if not tgz_files:
    print("\n❌ No se encontraron archivos .tgz")
    print(f"   Por favor sube 'lines.tgz' y 'ascii.tgz' a: {config.IAM_DIR}")
else:
    print(f"\n📦 Se han encontrado {len(tgz_files)} archivos comprimidos.")

    # Bucle para descomprimir TODOS los archivos .tgz encontrados
    for tgz in tgz_files:
        print(f"   ➡️ Extrayendo: {tgz.name}...")
        try:
            # Abrir en modo lectura (r:gz maneja .tgz y .tar.gz)
            with tarfile.open(tgz, 'r:gz') as tar:
                tar.extractall(config.IAM_DIR)
            print(f"      ✅ {tgz.name} extraído correctamente.")
        except Exception as e:
            print(f"      ❌ Error extrayendo {tgz.name}: {e}")

    # Verificar conteo final
    total_files = len(list(config.IAM_DIR.rglob('*')))
    print(f"\n✅ Proceso terminado. Total archivos en carpeta IAM: {total_files:,}")

In [ ]:
# ============================================================
# CELDA 8: Parser IAM
# ============================================================

def parse_iam():
    """Parsea lines.txt de IAM"""

    # Buscar GT file
    gt_paths = [
        config.IAM_DIR / 'lines.txt',
        config.IAM_DIR / 'ascii' / 'lines.txt'
    ]

    gt_file = None
    for path in gt_paths:
        if path.exists():
            gt_file = path
            break

    if not gt_file:
        raise FileNotFoundError("lines.txt no encontrado en IAM/")

    print(f"📄 Parseando: {gt_file}")

    samples = []
    with open(gt_file, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith('#'):
                continue

            parts = line.split(' ')
            if len(parts) < 9:
                continue

            line_id = parts[0]
            status = parts[1]
            text = ' '.join(parts[8:])

            if status != 'ok':
                continue

            # Path: a01-000u-00 → lines/a01/a01-000u/a01-000u-00.png
            id_parts = line_id.split('-')
            if len(id_parts) < 3:
                continue

            folder1 = id_parts[0]
            folder2 = f"{id_parts[0]}-{id_parts[1]}"
            img_name = f"{line_id}.png"

            img_path = config.IAM_DIR / 'lines' / folder1 / folder2 / img_name

            if img_path.exists():
                samples.append({
                    'image_path': str(img_path),
                    'text': text
                })

    return samples

print("Parseando IAM...")
all_samples = parse_iam()

print(f"\n✅ Total: {len(all_samples):,} muestras")
print(f"\n📝 Ejemplos:")
for s in random.sample(all_samples, min(5, len(all_samples))):
    text = s['text'][:60]
    print(f"   '{text}...'" if len(s['text']) > 60 else f"   '{text}'")

In [ ]:
# ============================================================
# CELDA 9: Split Dataset
# ============================================================

random.shuffle(all_samples)

total = len(all_samples)
train_split = int(0.8 * total)
val_split = int(0.9 * total)

train_samples = all_samples[:train_split]
val_samples = all_samples[train_split:val_split]
test_samples = all_samples[val_split:]

print(f"✅ Split:")
print(f"   Train: {len(train_samples):,}")
print(f"   Val:   {len(val_samples):,}")
print(f"   Test:  {len(test_samples):,}")

In [ ]:
# ============================================================
# CELDA 10: IAM Dataset Class
# ============================================================

class IAMDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Load image
        img = cv2.imread(sample['image_path'])
        if img is None:
            img = np.ones((64, 256, 3), dtype=np.uint8) * 255

        # RGB
        if len(img.shape) == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Transform
        if self.transform:
            img = self.transform(image=img)['image']

        # Encode
        text = sample['text']
        label = encode_text(text)
        label_length = len(label)

        return img, torch.LongTensor(label), label_length, text

print("✅ IAMDataset")

In [ ]:
# ============================================================
# CELDA 11: Transformaciones (MÁS AGRESIVAS)
# ============================================================

train_transform = A.Compose([
    A.Resize(config.IMG_HEIGHT, config.IMG_WIDTH),
    A.Rotate(limit=5, border_mode=cv2.BORDER_CONSTANT, value=255, p=0.5),
    A.Perspective(scale=(0.02, 0.08), p=0.3),
    A.ElasticTransform(alpha=50, sigma=7, p=0.2),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 7)),
        A.MotionBlur(blur_limit=(3, 7)),
    ], p=0.4),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(config.IMG_HEIGHT, config.IMG_WIDTH),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

def collate_fn(batch):
    images, labels, label_lengths, texts = zip(*batch)
    images = torch.stack(images, 0)

    max_label_len = max(label_lengths)
    labels_padded = []
    for label in labels:
        label_len = len(label)
        if label_len < max_label_len:
            padding = torch.zeros(max_label_len - label_len, dtype=torch.long)
            label_padded = torch.cat([label, padding])
        else:
            label_padded = label
        labels_padded.append(label_padded)

    labels_padded = torch.stack(labels_padded, 0)
    label_lengths = torch.LongTensor(label_lengths)

    return images, labels_padded, label_lengths, texts

print("✅ Transforms (agresivos)")

In [ ]:
# ============================================================
# CELDA 12: DataLoaders
# ============================================================

train_dataset = IAMDataset(train_samples, train_transform)
val_dataset = IAMDataset(val_samples, val_transform)
test_dataset = IAMDataset(test_samples, val_transform)

train_loader = DataLoader(
    train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
    num_workers=config.NUM_WORKERS, collate_fn=collate_fn,
    pin_memory=True, drop_last=True
)

val_loader = DataLoader(
    val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
    num_workers=config.NUM_WORKERS, collate_fn=collate_fn,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
    num_workers=config.NUM_WORKERS, collate_fn=collate_fn,
    pin_memory=True
)

print(f"✅ DataLoaders:")
print(f"   Train: {len(train_loader)} batches")
print(f"   Val:   {len(val_loader)} batches")

In [ ]:
# ============================================================
# CELDA 13: Modelo CRNN (IGUAL que impreso)
# ============================================================

class ImprovedCRNN(nn.Module):
    def __init__(self, img_height=64, num_classes=config.NUM_CLASSES,
                 hidden_size=512, num_lstm_layers=3):
        super(ImprovedCRNN, self).__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(True),

            nn.Conv2d(512, 512, (4, 3), padding=(0, 1)), nn.BatchNorm2d(512), nn.ReLU(True),
        )

        self.rnn = nn.LSTM(512, hidden_size, num_lstm_layers, bidirectional=True,
                          batch_first=True, dropout=0.3 if num_lstm_layers > 1 else 0)

        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

        self.dropout = nn.Dropout(0.3)
        self.linear1 = nn.Linear(hidden_size * 2, hidden_size)
        self.relu = nn.ReLU(True)
        self.linear2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        conv_out = self.cnn(x)
        conv_out = conv_out.squeeze(2).permute(0, 2, 1)
        rnn_out, _ = self.rnn(conv_out)
        rnn_out = self.dropout(rnn_out)
        output = self.linear1(rnn_out)
        output = self.relu(output)
        output = self.dropout(output)
        output = self.linear2(output)
        output = output.permute(1, 0, 2)
        return F.log_softmax(output, dim=2)

print("✅ CRNN")

---

## 🔄 TRANSFER LEARNING

1. Cargar pesos del modelo impreso
2. Congelar CNN (15 épocas)
3. Entrenar solo RNN + FC
4. Descongelar CNN y fine-tune completo

---

In [ ]:
# ============================================================
# CELDA 14: Transfer Learning - Cargar Modelo Impreso
# ============================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Crear modelo
model = ImprovedCRNN(
    img_height=config.IMG_HEIGHT,
    num_classes=config.NUM_CLASSES,
    hidden_size=config.HIDDEN_SIZE,
    num_lstm_layers=config.NUM_LSTM_LAYERS
)

# Verificar modelo impreso
if not config.PRINTED_MODEL.exists():
    print("\n❌ Modelo impreso no encontrado")
    print(f"   Esperado: {config.PRINTED_MODEL}")
    raise FileNotFoundError("Entrena primero el modelo impreso")

# Cargar pesos
print(f"\n📥 Cargando modelo impreso...")
checkpoint = torch.load(config.PRINTED_MODEL, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print("✅ Pesos cargados")

# CONGELAR CNN
if config.FREEZE_CNN:
    print(f"\n❄️  Congelando CNN hasta época {config.UNFREEZE_EPOCH}")
    for param in model.cnn.parameters():
        param.requires_grad = False

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n📊 Parámetros:")
print(f"   Total: {total:,}")
print(f"   Entrenables: {trainable:,} ({trainable/total*100:.1f}%)")
print(f"   Congelados: {total-trainable:,}")

In [ ]:
# ============================================================
# CELDA 15: Optimizer (CONSERVADOR)
# ============================================================

ctc_loss = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

steps_per_epoch = len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=config.LEARNING_RATE * 2,
    total_steps=config.MAX_EPOCHS * steps_per_epoch,
    pct_start=0.1,
    anneal_strategy='cos'
)

print("✅ Optimizer (LR bajo para fine-tuning)")

In [ ]:
# ============================================================
# CELDA 16-18: Checkpoint, EarlyStopping, Train/Val
# (COPIAR CELDAS 12-14 DEL NOTEBOOK IMPRESO)
# ============================================================

# CheckpointManager (igual que impreso)
class CheckpointManager:
    def __init__(self, checkpoint_dir, keep_last_n=5):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.keep_last_n = keep_last_n

    def save_checkpoint(self, epoch, model, optimizer, scheduler, history, is_best=False):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': history,
            'timestamp': datetime.now().isoformat()
        }

        checkpoint_path = self.checkpoint_dir / f'checkpoint_epoch_{epoch:04d}.pth'
        torch.save(checkpoint, checkpoint_path)

        if is_best:
            best_path = self.checkpoint_dir / 'best_model.pth'
            torch.save(checkpoint, best_path)
            print(f"✅ Mejor modelo guardado (época {epoch})")

        self._cleanup_old_checkpoints()
        return checkpoint_path

    def _cleanup_old_checkpoints(self):
        checkpoints = sorted(self.checkpoint_dir.glob('checkpoint_epoch_*.pth'))
        if len(checkpoints) > self.keep_last_n:
            for old_checkpoint in checkpoints[:-self.keep_last_n]:
                old_checkpoint.unlink()

    def load_latest_checkpoint(self, model, optimizer, scheduler):
        checkpoints = sorted(self.checkpoint_dir.glob('checkpoint_epoch_*.pth'))

        if not checkpoints:
            return 0, {}

        latest = checkpoints[-1]
        print(f"📥 Cargando: {latest.name}")

        checkpoint = torch.load(latest, map_location=device, weights_only=False)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

        return checkpoint['epoch'], checkpoint.get('history', {})

# EarlyStopping (igual que impreso)
class EarlyStopping:
    def __init__(self, patience=20, min_delta=0.0001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, epoch):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0
        return self.early_stop

# train_epoch (igual que impreso)
def train_epoch(model, dataloader, criterion, optimizer, scheduler, device, epoch):
    model.train()
    total_loss = 0
    all_predictions = []
    all_ground_truths = []

    pbar = tqdm(dataloader, desc=f'Época {epoch} - Train')

    for batch_idx, (images, labels, label_lengths, texts) in enumerate(pbar):
        images = images.to(device)
        labels = labels.to(device)
        label_lengths = label_lengths.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        input_lengths = torch.full(
            size=(images.size(0),),
            fill_value=outputs.size(0),
            dtype=torch.long
        ).to(device)

        loss = criterion(outputs, labels, input_lengths, label_lengths)

        if torch.isnan(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if batch_idx % 50 == 0:
            with torch.no_grad():
                _, preds = outputs.max(2)
                preds = preds.transpose(0, 1)
                for pred, text in zip(preds[:10], texts[:10]):
                    pred_text = decode_prediction(pred.cpu().numpy())
                    all_predictions.append(pred_text)
                    all_ground_truths.append(text)

        current_lr = optimizer.param_groups[0]['lr']
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'avg_loss': f'{total_loss/(batch_idx+1):.4f}',
            'lr': f'{current_lr:.6f}'
        })

    avg_loss = total_loss / len(dataloader)

    if all_predictions:
        avg_cer = sum(calculate_cer(p, t) for p, t in zip(all_predictions, all_ground_truths)) / len(all_predictions)
        avg_acc = sum(calculate_accuracy(p, t) for p, t in zip(all_predictions, all_ground_truths)) / len(all_predictions)
    else:
        avg_cer = 0.0
        avg_acc = 0.0

    return avg_loss, avg_cer, avg_acc

# validate (igual que impreso)
def validate(model, dataloader, criterion, device, epoch):
    model.eval()
    total_loss = 0
    all_predictions = []
    all_ground_truths = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc=f'Época {epoch} - Val')

        for images, labels, label_lengths, texts in pbar:
            images = images.to(device)
            labels = labels.to(device)
            label_lengths = label_lengths.to(device)

            outputs = model(images)

            input_lengths = torch.full(
                size=(images.size(0),),
                fill_value=outputs.size(0),
                dtype=torch.long
            ).to(device)

            loss = criterion(outputs, labels, input_lengths, label_lengths)

            if not torch.isnan(loss):
                total_loss += loss.item()

            _, preds = outputs.max(2)
            preds = preds.transpose(0, 1)

            for pred, text in zip(preds, texts):
                pred_text = decode_prediction(pred.cpu().numpy())
                all_predictions.append(pred_text)
                all_ground_truths.append(text)

            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / len(dataloader)
    avg_cer = sum(calculate_cer(p, t) for p, t in zip(all_predictions, all_ground_truths)) / len(all_predictions)
    avg_wer = sum(calculate_wer(p, t) for p, t in zip(all_predictions, all_ground_truths)) / len(all_predictions)
    avg_acc = sum(calculate_accuracy(p, t) for p, t in zip(all_predictions, all_ground_truths)) / len(all_predictions)

    return avg_loss, avg_cer, avg_wer, avg_acc, all_predictions, all_ground_truths

checkpoint_manager = CheckpointManager(config.CHECKPOINT_DIR, config.KEEP_LAST_N_CHECKPOINTS)
early_stopping = EarlyStopping(patience=config.EARLY_STOP_PATIENCE)

print("✅ Training funciones")

In [ ]:
# ============================================================
# CELDA 19: Training Loop CON DESCONGELACIÓN
# ============================================================

def train_with_unfreezing():
    start_epoch, history = checkpoint_manager.load_latest_checkpoint(
        model, optimizer, scheduler
    )

    if not history:
        history = {
            'train_loss': [], 'train_cer': [], 'train_acc': [],
            'val_loss': [], 'val_cer': [], 'val_wer': [], 'val_acc': [],
            'lr': []
        }

    best_val_cer = min(history['val_cer']) if history['val_cer'] else float('inf')

    print(f"\n{'='*70}")
    print(f"🚀 MANUSCRITO - TRANSFER LEARNING")
    print(f"{'='*70}")
    print(f"  Épocas: {start_epoch+1} → {config.MAX_EPOCHS}")
    print(f"  CNN congelado hasta época {config.UNFREEZE_EPOCH}")
    print(f"  Dataset: {len(train_samples):,} train | {len(val_samples):,} val")
    print(f"  Precisión esperada: 75-85% (manuscrito más difícil)")
    print(f"{'='*70}\n")

    for epoch in range(start_epoch + 1, config.MAX_EPOCHS + 1):

        # DESCONGELAR CNN
        if epoch == config.UNFREEZE_EPOCH and config.FREEZE_CNN:
            print(f"\n🔥 DESCONGELANDO CNN (época {epoch})")
            for param in model.cnn.parameters():
                param.requires_grad = True

            # Recrear optimizer con TODOS los params
            global optimizer
            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=config.LEARNING_RATE * 0.1,  # Aún más bajo
                weight_decay=config.WEIGHT_DECAY
            )
            print("✅ CNN descongelado, LR reducido")

        epoch_start = time.time()

        print(f"\n{'='*70}")
        print(f"📅 ÉPOCA {epoch}/{config.MAX_EPOCHS}")
        print(f"{'='*70}")

        # Train
        train_loss, train_cer, train_acc = train_epoch(
            model, train_loader, ctc_loss, optimizer, scheduler, device, epoch
        )

        # Val
        val_loss, val_cer, val_wer, val_acc, preds, gts = validate(
            model, val_loader, ctc_loss, device, epoch
        )

        epoch_time = time.time() - epoch_start

        # History
        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(train_loss)
        history['train_cer'].append(train_cer)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_cer'].append(val_cer)
        history['val_wer'].append(val_wer)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        # Print
        print(f"\n📊 ÉPOCA {epoch}:")
        print(f"{'─'*70}")
        print(f"  🔹 TRAIN → Loss: {train_loss:.4f} | CER: {train_cer:.4f} | ACC: {train_acc:.4f}")
        print(f"  🔸 VAL   → Loss: {val_loss:.4f} | CER: {val_cer:.4f} | WER: {val_wer:.4f} | ACC: {val_acc:.4f}")
        print(f"  ⏱️  Tiempo: {epoch_time/60:.1f} min | LR: {current_lr:.7f}")

        # Ejemplos
        print(f"\n📝 EJEMPLOS:")
        for idx in random.sample(range(len(preds)), min(3, len(preds))):
            match = "✅" if preds[idx] == gts[idx] else "❌"
            gt_text = gts[idx][:40] + '...' if len(gts[idx]) > 40 else gts[idx]
            pred_text = preds[idx][:40] + '...' if len(preds[idx]) > 40 else preds[idx]
            print(f"  {match} GT:   '{gt_text}'")
            print(f"     Pred: '{pred_text}'")

        # Save
        is_best = val_cer < best_val_cer
        if is_best:
            best_val_cer = val_cer

        if epoch % config.SAVE_EVERY_N_EPOCHS == 0 or is_best:
            checkpoint_manager.save_checkpoint(
                epoch, model, optimizer, scheduler, history, is_best
            )
            print(f"💾 Checkpoint guardado")

        # Early stop
        if early_stopping(val_loss, epoch):
            print(f"\n⚠️ EARLY STOP época {epoch}")
            break

    print(f"\n{'='*70}")
    print("🎉 MANUSCRITO COMPLETADO")
    print(f"{'='*70}")
    print(f"  Mejor Val CER: {best_val_cer:.4f} ({(1-best_val_cer)*100:.1f}% precisión)")
    print(f"  Total épocas: {len(history['val_loss'])}")

    return history

print("✅ Loop con descongelación definido")

In [ ]:
# ============================================================
# CELDA 20: ENTRENAR MANUSCRITO
# ============================================================

print("\n" + "="*70)
print("⚡ MANUSCRITO - TRANSFER LEARNING")
print("="*70)
print("\n⚠️ IMPORTANTE:")
print("  - Tardará 30-50 horas en Colab Free")
print("  - Se guardará cada 5 épocas")
print("  - CNN congelado hasta época", config.UNFREEZE_EPOCH)
print("  - Precisión esperada: 75-85%")
print("\n")

history = train_with_unfreezing()

print("\n✅ Listo!")

In [ ]:
# ============================================================
# CELDA 21: Evaluación Test
# ============================================================

print("\n" + "="*70)
print("📊 EVALUACIÓN TEST - MANUSCRITO")
print("="*70 + "\n")

best_checkpoint = torch.load(config.CHECKPOINT_DIR / 'best_model.pth', map_location=device, weights_only=False)
model.load_state_dict(best_checkpoint['model_state_dict'])
print(f"✅ Mejor modelo cargado (época {best_checkpoint['epoch']})")

test_loss, test_cer, test_wer, test_acc, test_preds, test_gts = validate(
    model, test_loader, ctc_loss, device, epoch=0
)

print(f"\n{'='*70}")
print(f"🎯 RESULTADOS TEST MANUSCRITO:")
print(f"{'='*70}")
print(f"  Loss: {test_loss:.4f}")
print(f"  CER: {test_cer:.4f} ({(1-test_cer)*100:.2f}% precisión)")
print(f"  WER: {test_wer:.4f}")
print(f"  Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"{'='*70}\n")

# Guardar resultados
results = {
    'test_loss': test_loss,
    'test_cer': test_cer,
    'test_wer': test_wer,
    'test_acc': test_acc,
    'total_samples': len(test_preds),
    'correct': sum(1 for p, t in zip(test_preds, test_gts) if p == t),
    'timestamp': datetime.now().isoformat()
}

with open(config.RESULTS_DIR / 'test_results_handwriting.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"✅ Resultados: {config.RESULTS_DIR / 'test_results_handwriting.json'}")

In [ ]:
# ============================================================
# CELDA 22: Gráficas
# ============================================================

def plot_training_history(history):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    epochs = range(1, len(history['train_loss']) + 1)

    axes[0, 0].plot(epochs, history['train_loss'], 'o-', label='Train')
    axes[0, 0].plot(epochs, history['val_loss'], 's-', label='Val')
    axes[0, 0].set_xlabel('Época')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(epochs, history['train_cer'], 'o-', label='Train')
    axes[0, 1].plot(epochs, history['val_cer'], 's-', label='Val')
    axes[0, 1].set_xlabel('Época')
    axes[0, 1].set_ylabel('CER')
    axes[0, 1].set_title('Character Error Rate')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[0, 2].plot(epochs, history['train_acc'], 'o-', label='Train')
    axes[0, 2].plot(epochs, history['val_acc'], 's-', label='Val')
    axes[0, 2].set_xlabel('Época')
    axes[0, 2].set_ylabel('Accuracy')
    axes[0, 2].set_title('Exact Match Accuracy')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)

    axes[1, 0].plot(epochs, history['val_wer'], 's-', color='green')
    axes[1, 0].set_xlabel('Época')
    axes[1, 0].set_ylabel('WER')
    axes[1, 0].set_title('Word Error Rate')
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(epochs, history['lr'], 'o-', color='red')
    axes[1, 1].set_xlabel('Época')
    axes[1, 1].set_ylabel('LR')
    axes[1, 1].set_title('Learning Rate')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3)

    axes[1, 2].axis('off')
    summary = f"""
    MANUSCRITO - TRANSFER LEARNING
    {'─'*30}

    Épocas: {len(epochs)}
    Mejor época: {np.argmin(history['val_cer']) + 1}

    Mejor Train CER: {min(history['train_cer']):.4f}
    Mejor Val CER: {min(history['val_cer']):.4f}
    Mejor Val ACC: {max(history['val_acc']):.4f}

    Final Train CER: {history['train_cer'][-1]:.4f}
    Final Val CER: {history['val_cer'][-1]:.4f}
    """
    axes[1, 2].text(0.1, 0.5, summary, fontsize=10, family='monospace', va='center')

    plt.tight_layout()
    plt.savefig(config.RESULTS_DIR / 'training_history_handwriting.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Gráfica: {config.RESULTS_DIR / 'training_history_handwriting.png'}")

# Cargar history
best_checkpoint = torch.load(config.CHECKPOINT_DIR / 'best_model.pth', map_location=device, weights_only=False)

if 'history' in best_checkpoint:
    history = best_checkpoint['history']
    plot_training_history(history)
else:
    print("⚠️ No history en checkpoint")

In [ ]:
# ============================================================
# CELDA 23: Guardar Modelo Final Manuscrito
# ============================================================

final_path = config.MODEL_DIR / 'ocr_model_handwriting_final.pth'

torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'img_height': config.IMG_HEIGHT,
        'img_width': config.IMG_WIDTH,
        'num_classes': config.NUM_CLASSES,
        'hidden_size': config.HIDDEN_SIZE,
        'num_lstm_layers': config.NUM_LSTM_LAYERS,
    },
    'vocab': {
        'chars': config.CHARS,
        'char_to_idx': config.CHAR_TO_IDX,
        'idx_to_char': config.IDX_TO_CHAR,
    },
    'test_results': results,
    'timestamp': datetime.now().isoformat()
}, final_path)

print(f"\n✅ MODELO MANUSCRITO GUARDADO:")
print(f"   {final_path}")
print(f"\n📊 Rendimiento:")
print(f"   Test CER: {test_cer:.4f} ({(1-test_cer)*100:.1f}% precisión)")
print(f"   Test ACC: {test_acc:.2%}")
print(f"\n🎉 ¡Listo para usar en la app!")